# Dutch North Sea Decommissioning — Exploratory Notebook

Seven sections, built and committed one at a time.

| Section | Purpose |
|---|---|
| 1 | Fetch raw borehole data from NLOG API |
| 2 | Inspect shape, dtypes, uniques, nulls |
| 3 | Filter offshore, split Group A / B / ghost |
| 4 | Operator analysis on Group A |
| 5 | Production crossref — informal cessation |
| 6 | Join and summarise |
| 7 | Sense check against data diary |

**Data dirs** (`netherlands/data/`) are gitignored. Raw JSON and all CSVs  
live only on your local machine unless explicitly exported.

---
## Section 1 — Fetch

Uses `requests.Session` to:
1. `GET` the datacenter overview page — sets the required session cookie.
2. `POST` (empty body) to the boreholes endpoint — returns all 6,723 wells.

Saves raw JSON to `netherlands/data/raw/nlog_boreholes_raw.json`.

In [ ]:
import json
import logging
from pathlib import Path

import requests

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Paths — notebook lives in netherlands/notebooks/, data two levels over.
# Path.cwd() resolves to the notebook directory when launched normally.
# ---------------------------------------------------------------------------
NOTEBOOKS_DIR = Path.cwd()
NETHERLANDS_ROOT = NOTEBOOKS_DIR.parent          # netherlands/
DATA_RAW = NETHERLANDS_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED = NETHERLANDS_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR = NETHERLANDS_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSON_PATH = DATA_RAW / "nlog_boreholes_raw.json"

print(f"Project root : {NETHERLANDS_ROOT}")
print(f"Raw JSON     : {RAW_JSON_PATH}")

In [ ]:
# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

_SEED_URL = "https://www.nlog.nl/datacenter/brh-overview"
_BOREHOLES_URL = "https://www.nlog.nl/nlog-mapviewer/rest/brh/boreholes"
_TIMEOUT = 60  # seconds
_EXPECTED_COUNT = 6_723
_EXPECTED_FIELDS = {
    "boreholeDbk",
    "boreholeName",
    "shortName",
    "clientOrgName",
    "legalOwnerName",
    "statusDescription",
    "resultCode",
    "onOffshore",
    "startDate",
    "endDate",
    "confidentialityDate",
    "blockCd",
}

In [ ]:
def seed_session() -> requests.Session:
    """Open a session and hit the datacenter overview page to set the required cookie."""
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; research/1.0)"})
    resp = session.get(_SEED_URL, timeout=_TIMEOUT)
    resp.raise_for_status()
    logger.info("Seed GET %s → %s (cookies: %s)", _SEED_URL, resp.status_code, list(session.cookies.keys()))
    return session


def fetch_boreholes(session: requests.Session) -> list:
    """POST to the boreholes endpoint and return the parsed JSON array."""
    resp = session.post(_BOREHOLES_URL, timeout=_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    logger.info("Boreholes POST → %s records", len(data))
    return data


def validate_record_count(data: list) -> None:
    """Warn loudly if the record count differs significantly from the expected total."""
    count = len(data)
    delta = abs(count - _EXPECTED_COUNT)
    if delta > 50:
        logger.warning(
            "COUNT MISMATCH — got %d records, expected ~%d (delta %d). "
            "Stop and flag before proceeding.",
            count, _EXPECTED_COUNT, delta,
        )
    else:
        logger.info("Record count %d — within 50 of expected %d ✓", count, _EXPECTED_COUNT)


def validate_field_names(record: dict) -> None:
    """Check that every expected field is present in the first record."""
    actual = set(record.keys())
    missing = _EXPECTED_FIELDS - actual
    extra = actual - _EXPECTED_FIELDS
    if missing:
        logger.warning("MISSING FIELDS: %s", missing)
    if extra:
        logger.info("Extra fields not in spec (note for data diary): %s", extra)
    if not missing:
        logger.info("All expected fields present ✓")


def save_raw(data: list, path: Path) -> None:
    """Write the raw JSON array to disk."""
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2))
    logger.info("Saved %d records to %s", len(data), path)

In [ ]:
# ---------------------------------------------------------------------------
# Run Section 1
# ---------------------------------------------------------------------------

session = seed_session()
raw_data = fetch_boreholes(session)

validate_record_count(raw_data)
validate_field_names(raw_data[0])

print("\n--- First record ---")
print(json.dumps(raw_data[0], indent=2, ensure_ascii=False))

save_raw(raw_data, RAW_JSON_PATH)

---
## Section 2 — Inspect

Load raw JSON into a pandas DataFrame and verify:
- Shape, column names, dtypes
- All unique `statusDescription` values (expected: 7 + null)
- All unique `resultCode` values (expected: 16 + null)
- Null counts per column
- Offshore vs onshore split
- First five rows

Flag anything that deviates from the spec before proceeding to Section 3.

In [ ]:
import pandas as pd

# Expected values from the data spec — used to detect drift
_EXPECTED_STATUSES = {
    "Plugged and abandoned",
    "Producing/Injecting",
    "Monitoring",
    "Suspended",
    "Closed-In",
    "Sidetracked",
    None,  # ghost wells
}

_EXPECTED_RESULT_CODES = {
    "GAS", "OIL", "OAG", "GOS", "GSS", "OLS",  # hydrocarbon — Group A candidates
    "DRY", "SHW", "INC", "UNK", "WTR", "CNS",  # non-hydrocarbon
    "INJ", "STO", "GNS", "OGS",                 # injection / storage
    None,
}

In [ ]:
def load_raw_json(path: Path) -> pd.DataFrame:
    """Load the raw borehole JSON into a DataFrame."""
    data = json.loads(path.read_text())
    df = pd.DataFrame(data)
    logger.info("Loaded DataFrame: %d rows × %d cols", *df.shape)
    return df


def convert_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Convert Unix-millisecond date columns to pandas datetime (UTC)."""
    for col in ("startDate", "endDate", "confidentialityDate"):
        df[col] = pd.to_datetime(df[col], unit="ms", utc=True, errors="coerce")
    return df


def print_shape_and_dtypes(df: pd.DataFrame) -> None:
    """Print the DataFrame shape and column dtypes."""
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
    print("Column dtypes:")
    print(df.dtypes.to_string())


def print_unique_values(df: pd.DataFrame, col: str, expected: set) -> None:
    """Print all unique values for a column and flag any that differ from the spec."""
    actual = set(df[col].unique())  # includes NaN as float nan
    # Normalise: pandas represents None/null as np.nan for object cols
    actual_normalised = {None if pd.isna(v) else v for v in actual}
    unexpected = actual_normalised - expected
    missing = expected - actual_normalised

    counts = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="count")
    print(f"\n--- {col} ({len(actual_normalised)} unique values) ---")
    print(counts.to_string(index=False))

    if unexpected:
        logger.warning("UNEXPECTED values in %s: %s — flag before proceeding", col, unexpected)
    if missing:
        logger.info("Values in spec but absent from data (may be normal): %s", missing)
    if not unexpected:
        logger.info("%s values match spec ✓", col)


def print_null_counts(df: pd.DataFrame) -> None:
    """Print null count and percentage for every column."""
    nulls = df.isnull().sum()
    pct = (nulls / len(df) * 100).round(1)
    summary = pd.DataFrame({"null_count": nulls, "null_pct": pct})
    summary = summary[summary["null_count"] > 0].sort_values("null_count", ascending=False)
    print("\n--- Null counts (columns with at least one null) ---")
    print(summary.to_string())


def print_on_offshore_split(df: pd.DataFrame) -> None:
    """Print count of onshore vs offshore records."""
    split = df["onOffshore"].value_counts(dropna=False)
    print("\n--- onOffshore split ---")
    print(split.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Run Section 2
# ---------------------------------------------------------------------------

df = load_raw_json(RAW_JSON_PATH)
df = convert_dates(df)

print_shape_and_dtypes(df)
print_unique_values(df, "statusDescription", _EXPECTED_STATUSES)
print_unique_values(df, "resultCode", _EXPECTED_RESULT_CODES)
print_null_counts(df)
print_on_offshore_split(df)

print("\n--- First five rows (key columns) ---")
key_cols = ["boreholeName", "clientOrgName", "statusDescription",
            "resultCode", "onOffshore", "startDate", "endDate", "blockCd"]
print(df[key_cols].head().to_string(index=False))